In [1]:
!pip install ms-swift decord 'qwen_vl_utils>=0.0.14'

In [2]:
import os
from swift.infer_engine import TransformersEngine, RequestConfig, InferRequest
model = 'Qwen/Qwen3.5-2B'

In [5]:
os.environ['USE_HF'] = "1"
engine = TransformersEngine(model, torch_dtype="float16", max_batch_size=2)

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
[INFO:swift] Setting ROOT_IMAGE_DIR: None. You can adjust this hyperparameter through the environment variable: `ROOT_IMAGE_DIR`.
[INFO:swift] Setting QWENVL_BBOX_FORMAT: legacy. You can adjust this hyperparameter through the environment variable: `QWENVL_BBOX_FORMAT`.


In [6]:
user_query = "Stop! Who approacheth the Bridge of Death must answer me these questions three, ere the other side he see. What is your name? What is your quest? What is the air-speed velocity of an unladen swallow?"

In [7]:
infer_request = [InferRequest(messages=[{
    'role': 'user',
    'content': user_query
}])]

In [8]:
temperature = [0.0, 1.0, 10.0, 1000.0]

In [9]:
for T in temperature:
  request_config = RequestConfig(max_tokens=512, temperature=T, top_k=248320, top_p=1)

  resp_list = engine.infer(infer_request, request_config)

  response = resp_list[0].choices[0].message.content

  print(f'===== Temperature: {T} =====\n')
  print(response + "\n\n")

[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linea

===== Temperature: 0.0 =====

<think>

</think>

The bridge is not a physical structure you can walk across; it is a metaphor for the final moments of life, often associated with the concept of *Memento Mori* (remembering you must die). The "Bridge of Death" is a famous image from Dante's *Inferno*, where souls are judged before crossing over to the afterlife.

Regarding your questions:

1.  **Who approaches the Bridge of Death?**
    It is **no one**. The bridge is a symbolic representation of the human condition and the inevitability of death. It is not a place where specific people approach; rather, it is the threshold that *all* humans eventually cross. In the context of the poem, it is the moment of reckoning where the soul confronts the reality of mortality.

2.  **What is your name?**
    There is no single name. The bridge is a universal symbol for the human experience. If you are asking about the specific soul in the poem, it is **Dante**, the pilgrim who seeks to understand t

## Wnioski

* Parametr temperatury ma bezpośredni i kluczowy wpływ na przewidywalność i powtarzalność odpowiedzi generowanych przez model językowy lub na ich kreatywność.   


* Dla najniższej temperatury (0.0) odpowiedź modelu jest bardzo dokładna i precyzyjna. Ten poziom temperatury to dobry wybór, gdy pożądamy konkretnej informacji i zależy nam na powtarzalności w odpowiedzi.   


* Zwiększenie temperatury do (1.0) powoduje, że odpowiedzi są bardziej kreatywne i urozmaicone, ale wciąż jeszcze merytoryczne i zrozumiałe. To dobra warość, jeżeli oczekuje się zbalansowanej, bogatszej językowo odpowiedzi, gdzie potrzebna jest odrobina pomysłowości, nie tracąc przy tym poprawnej logiki. Co istotne, ta wartość temperatury nie zmienia oryginalnego rozkładu prawdopodobieństwa, ponieważ we wzorze funkcji softmax dzielimy logity przez temperaturę, a w tym wypadku przez 1, czyli temperatura nie wpływa na oryginalny rozkład prawdopodobieństwa.


* Przy bardzo wysokich ustawieniach (10.0) użyteczność modelu bardzo mocno spada. Odpowiedzi stają się mieszaniną losowych znaków i przypadkowych słów. Taka odpowiedź staje się totalnie bezużyteczna. Winika to z faktu, że rozkład pradopodobieństwa w modelu jest już tak bardzo spłaszczony, że tokeny są dopierane w sposób przypadkowy.


* Temperatura (1000.0) podobnie jak w przypadku wartości 10.0 generuje losowy ciąg znaków. Różnica między wartością 10 a 1000 jest taka, że nie ma tu już prawie w ogóle słów, jest jedynie bezsensowny ciąg znaków.

* Widać więć, że istnieje pewna wartośc temperatury, powyżej której jej zwiększanie nie ma już żadnego znaczenia, ponieważ generowane odpowiedzi są całkowicie bezsensowne.

## Ważne ustawienia

* Na potrzeby zbadania wpływu samej temperatury na generowanie odpowiedzi, wyłączyłem top_k i top_p. Są to filtry które ograniczają ilość branych pod uwagę tokenów ze słownika. Powoduje to, że na temperaturze 10, czy 1000 wygenerowane odpowiedzi brzmią bardziej naturalnie niż by brzmiały bez użycia top_k i top_p, przez co testy nie ilustrują wpływu samej temperatury. Ustawiłem top_k na wartość odpowaidającą długości słownika w tym modelu, po to, aby nie były odcinane żadne tokeny, a top_p ustawiłem na 1 co oznacza 100%.


* Domyślnie tryb myślenia (enable_thinging) jest wyłączony. Zostawiłem to tak, z dwóch głownych powodów:

1. Włączone myślenie działa jak filtr, który koryguje losowość i może dać identyczne odpowiedzi końcowe zarówno przy niskiej, jak i wysokiej temperaturze. Wyłączenie tej opcji pozwoli Ci precyzyjnie i bez zakłóceń zaobserwować, jak sama temperatura wpływa na kreatywność i dobór słów przez model.


2. Poza tym generowanie długiego łańcucha myślowego zużywa cenne tokeny, co mogłoby doprowadzić do przedwczesnego ucięcia właściwej odpowiedzi.